## Setup and Dependencies

In [2]:
import io
import os
from pathlib import Path
from typing import Any, cast

import cv2
import fitz  # PyMuPDF  # type: ignore[import-untyped]
import numpy as np
from dotenv import load_dotenv
from PIL import Image

In [3]:
# Load environment variables
load_dotenv()

# Configuration
PDF_RENDER_DPI = 200  # DPI for PDF rendering
PADDING_PIXELS = 4  # Extra padding around crop

# Serverless compatibility: Only save to disk if explicitly enabled
SAVE_CROPS = os.getenv("SAVE_CROPS", "false").lower() == "true"

print(f"✓ Configuration loaded")
print(f"  PDF Render DPI: {PDF_RENDER_DPI}")
print(f"  Padding: {PADDING_PIXELS}px")
print(f"  Save crops to tmp/: {'Enabled' if SAVE_CROPS else 'Disabled (serverless mode)'}")

✓ Configuration loaded
  PDF Render DPI: 200
  Padding: 4px
  Save crops to tmp/: Enabled


## Helper Functions

In [4]:
def convert_pdf_to_image_bytes(file_path: str, dpi: int = 200) -> bytes:
    """
    Convert PDF first page to PNG image bytes, or return original bytes if already an image.
    
    Args:
        file_path: Path to PDF or image file
        dpi: DPI for PDF rendering (default: 200)
    
    Returns:
        PNG image bytes
    """
    file_ext = Path(file_path).suffix.lower()
    
    if file_ext == ".pdf":
        # Read PDF file
        with open(file_path, "rb") as f:
            file_bytes = f.read()
        
        # Extract first page of PDF as PNG image bytes
        pdf_document: Any = fitz.open(stream=file_bytes, filetype="pdf")
        if pdf_document.page_count == 0:
            raise ValueError("PDF has no pages")
        
        # Render first page to image at specified DPI
        page: Any = pdf_document[0]
        pix: Any = page.get_pixmap(dpi=dpi)
        png_bytes: bytes = pix.tobytes("png")
        return png_bytes
    else:
        # Already an image, read and return as-is
        with open(file_path, "rb") as f:
            return f.read()


def polygon_to_bbox(points: list[float]) -> tuple[float, float, float, float]:
    """Convert polygon [x1,y1,x2,y2,...] -> (min_x, min_y, max_x, max_y)."""
    xs = points[::2]
    ys = points[1::2]
    return min(xs), min(ys), max(xs), max(ys)


def crop_image_region(
    image_bytes: bytes, bbox: tuple[float, float, float, float], padding: int = 0
) -> tuple[bytes, Any]:
    """
    Crop region from image bytes.
    
    Args:
        image_bytes: Image file bytes
        bbox: Bounding box (min_x, min_y, max_x, max_y)
        padding: Padding in pixels
    
    Returns:
        Tuple of (PNG bytes, numpy array)
    """
    # Load image from bytes
    im = Image.open(io.BytesIO(image_bytes)).convert("RGBA")
    min_x, min_y, max_x, max_y = bbox
    
    box = (
        max(int(min_x) - padding, 0),
        max(int(min_y) - padding, 0),
        min(int(max_x) + padding, im.width),
        min(int(max_y) + padding, im.height),
    )
    crop = im.crop(box)
    
    # Convert to PNG bytes
    img_byte_arr = io.BytesIO()
    crop.save(img_byte_arr, format='PNG')
    png_bytes = img_byte_arr.getvalue()
    
    # Convert to OpenCV format (numpy array)
    nparr = np.frombuffer(png_bytes, np.uint8)
    img_array: Any = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    
    return png_bytes, img_array

## Crop Images using OpenCV

In [ ]:
def crop_signatures_opencv(
    directory_path: str = "../tmp",
    output_dir: str = "../cropped"
) -> dict[str, Any]:
    """
    Crop signatures from images in a directory using OpenCV contour detection.
    Uses the same fallback logic as function_app.py _extract_signatures().
    
    Args:
        directory_path: Path to directory containing images (default: "../tmp")
        output_dir: Directory to save cropped signatures (default: "../cropped")
    
    Returns:
        Dictionary mapping filenames to extraction results
    """
    results: dict[str, Any] = {}
    supported_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}
    
    directory = Path(directory_path)
    if not directory.exists():
        raise FileNotFoundError(f"Directory not found: {directory_path}")
    
    # Create output directory
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    files = [f for f in directory.iterdir() if f.suffix.lower() in supported_extensions]
    
    print(f"\nFound {len(files)} file(s) to process with OpenCV\n")
    print("="*60)
    
    for idx, file in enumerate(files, 1):
        print(f"\n[{idx}/{len(files)}] Processing: {file.name}")
        print("-"*60)
        
        try:
            # Read image
            image: Any = cv2.imread(str(file))
            if image is None:
                raise ValueError(f"Failed to load image: {file.name}")
            
            # Convert to grayscale
            gray: Any = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  # type: ignore[assignment]
            
            # Apply Gaussian blur to reduce noise
            blurred: Any = cv2.GaussianBlur(gray, (5, 5), 0)  # type: ignore[arg-type]
            
            # Apply adaptive thresholding
            thresh: Any = cv2.adaptiveThreshold(  # type: ignore[assignment]
                blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2  # type: ignore[arg-type]
            )
            
            # Find contours
            contours_result: Any = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)  # type: ignore[arg-type]
            contours: Any = contours_result[0]  # type: ignore[assignment]
            
            # Filter contours by area and aspect ratio (typical signature characteristics)
            min_area = 500
            signatures: list[Any] = []
            
            for contour in contours:  # type: ignore[attr-defined]
                area = cv2.contourArea(contour)  # type: ignore[arg-type]
                if area < min_area:
                    continue
                
                x, y, w, h = cv2.boundingRect(contour)  # type: ignore[arg-type]
                aspect_ratio = w / float(h) if h > 0 else 0
                
                # Signatures typically have aspect ratio between 1.5 and 5.0
                if 1.5 <= aspect_ratio <= 5.0 and w > 50 and h > 20:
                    # Extract signature region with some padding
                    padding = 10
                    x1 = max(0, x - padding)
                    y1 = max(0, y - padding)
                    x2 = min(image.shape[1], x + w + padding)
                    y2 = min(image.shape[0], y + h + padding)
                    
                    signature_img = image[y1:y2, x1:x2]
                    signatures.append({
                        'image': signature_img,
                        'bbox': (x1, y1, x2, y2),
                        'area': area,
                        'aspect_ratio': aspect_ratio
                    })
            
            # Sort by area (largest first) and keep top 3
            signatures.sort(key=lambda s: s['area'], reverse=True)
            top_signatures = signatures[:3]
            
            # Save cropped signatures
            file_stem = file.stem
            saved_files: list[str] = []
            for sig_idx, sig in enumerate(top_signatures, 1):
                output_filename = f"{file_stem}_sig_{sig_idx}.png"
                output_file = output_path / output_filename
                cv2.imwrite(str(output_file), sig['image'])  # type: ignore[arg-type]
                saved_files.append(output_filename)
                print(f"  Saved: {output_filename} (size: {sig['image'].shape[1]}x{sig['image'].shape[0]}, "
                      f"aspect_ratio: {sig['aspect_ratio']:.2f})")
            
            results[file.name] = {
                'file': str(file),
                'signatures_found': len(top_signatures),
                'total_contours': len(signatures),
                'saved_files': saved_files,
                'output_dir': str(output_path)
            }
            print(f"✓ Success: {len(top_signatures)} signature(s) extracted from {len(signatures)} candidates")
            
        except Exception as e:
            print(f"✗ Error: {str(e)}")
            results[file.name] = {
                'file': str(file),
                'error': str(e),
                'signatures_found': 0
            }
    
    print("\n" + "="*60)
    print("OPENCV BATCH PROCESSING COMPLETE")
    print("="*60)
    
    total_signatures = sum(
        r.get('signatures_found', 0) for r in results.values()
    )
    successful = sum(
        1 for r in results.values() if 'error' not in r
    )
    
    print(f"\nProcessed: {len(files)} file(s)")
    print(f"Successful: {successful}")
    print(f"Failed: {len(files) - successful}")
    print(f"Total signatures extracted: {total_signatures}")
    print(f"Output directory: {output_dir}")
    
    return results


# Example: Crop signatures from tmp directory using OpenCV
# Uncomment to run:
opencv_results = crop_signatures_opencv("../tmp/s0-nocenter-100/ADI1", "../tmp/s0-nocenter-100/CROP")


Found 7 file(s) to process with OpenCV


[1/7] Processing: valid_id1_sig_1_1763187513397_figure_1_p1_1_1763187635176.png
------------------------------------------------------------
  Saved: valid_id1_sig_1_1763187513397_figure_1_p1_1_1763187635176_sig_1.png (size: 152x70, aspect_ratio: 2.58)
  Saved: valid_id1_sig_1_1763187513397_figure_1_p1_1_1763187635176_sig_2.png (size: 114x67, aspect_ratio: 1.88)
✓ Success: 2 signature(s) extracted from 2 candidates

[2/7] Processing: valid_id2_sig_1_1763187520058_CustomerSignature_p1_1_1763187654447.png
------------------------------------------------------------
✓ Success: 0 signature(s) extracted from 0 candidates

[3/7] Processing: valid_id3_sig_1_1763187536532_CustomerSignature_p1_1_1763187673561.png
------------------------------------------------------------
✓ Success: 0 signature(s) extracted from 0 candidates

[4/7] Processing: valid_id4_sig_1_1763187543878_CustomerSignature_p1_1_1763187689331.png
---------------------------------------

## Upscaling images using OpenCV

LapSRN (multi-scale, good for text edges)

In [25]:
def upscale_image(
    input_path: str,
    output_path: str,
    model_path: str = "LapSRN_x2.pb",
    scale: int = 2,
) -> dict[str, Any]:
    """
    Upscale a single blurry image using OpenCV's DNN Super Resolution.
    
    Args:
        input_path: Path to input image file
        output_path: Path to save upscaled image
        model_path: Path to pre-trained model file (download from OpenCV model zoo)
        scale: Upscaling factor (2, 4, or 8)
    
    Returns:
        Dictionary with:
            - success: Whether operation succeeded
            - input_path: Input file path
            - output_path: Output file path
            - input_size: Original image dimensions (width, height)
            - output_size: Upscaled image dimensions (width, height)
            - scale_factor: Upscaling factor used
            - message: Success or error message
    
    Raises:
        FileNotFoundError: If input image or model file not found
        ValueError: If scale factor is invalid
    
    Example:
        >>> result = upscale_image("valid_id2.png", "document_upscaled.png")
        >>> print(result['message'])
    """
    try:
        # Validate scale factor
        valid_scales = [2, 4, 8]
        if scale not in valid_scales:
            raise ValueError(f"Invalid scale factor. Must be one of {valid_scales}")
        
        # Check if input file exists
        if not os.path.exists(input_path):
            raise FileNotFoundError(f"Input image not found: {input_path}")
        
        # Check if model file exists
        if not os.path.exists(model_path):
            raise FileNotFoundError(
                f"Model file not found: {model_path}\n"
                f"Download from: https://github.com/opencv/opencv_contrib/tree/master/modules/dnn_superres"
            )
        
        # Load input image
        img_result = cv2.imread(input_path)
        if img_result is None:
            raise ValueError(f"Failed to load image: {input_path}")
        
        img = cast(np.ndarray[Any, np.dtype[np.uint8]], img_result)
        shape_attr: Any = getattr(img, "shape", (0, 0, 0))
        input_shape = cast(tuple[int, int, int], shape_attr)
        input_height: int = input_shape[0]
        input_width: int = input_shape[1]
        
        # Create SR object (type: ignore for OpenCV contrib module)
        sr_obj: Any = cv2.dnn_superres.DnnSuperResImpl_create()  # type: ignore[attr-defined]
        sr = cast(Any, sr_obj)
        sr.readModel(model_path)
        sr.setModel("lapsrn", scale)
        
        # Upscale image
        upscaled_result: Any = sr.upsample(img)
        upscaled = cast(np.ndarray[Any, np.dtype[np.uint8]], upscaled_result)
        
        # Extract dimensions before save
        upscaled_shape_attr: Any = getattr(upscaled, "shape", (0, 0, 0))
        output_shape = cast(tuple[int, int, int], upscaled_shape_attr)
        output_height: int = output_shape[0]
        output_width: int = output_shape[1]
        
        # Save result
        cv2.imwrite(output_path, upscaled)
        
        return {
            "success": True,
            "input_path": input_path,
            "output_path": output_path,
            "input_size": (input_width, input_height),
            "output_size": (output_width, output_height),
            "scale_factor": scale,
            "message": f"Successfully upscaled image from {input_width}x{input_height} to {output_width}x{output_height}",
        }
    
    except Exception as e:
        return {
            "success": False,
            "input_path": input_path,
            "output_path": output_path,
            "input_size": None,
            "output_size": None,
            "scale_factor": scale,
            "message": f"Error: {str(e)}",
        }


def upscale_images_batch(
    directory_path: str = "../tmp",
    output_dir: str = "../tmp/upscaled",
    model_path: str = "LapSRN_x2.pb",
    scale: int = 2,
) -> dict[str, Any]:
    """
    Batch upscale all images in a directory using OpenCV's DNN Super Resolution.
    
    Args:
        directory_path: Path to directory containing images (default: "../tmp")
        output_dir: Directory to save upscaled images (default: "../tmp/upscaled")
        model_path: Path to pre-trained model file (default: "LapSRN_x2.pb")
        scale: Upscaling factor - 2, 4, or 8 (default: 2)
    
    Returns:
        Dictionary mapping filenames to upscale results
    
    Example:
        >>> results = upscale_images_batch("../tmp", "../tmp/upscaled", scale=2)
        >>> print(f"Processed {len(results)} images")
    """
    results: dict[str, Any] = {}
    supported_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}
    
    directory = Path(directory_path)
    if not directory.exists():
        raise FileNotFoundError(f"Directory not found: {directory_path}")
    
    # Create output directory
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Check if model file exists
    if not os.path.exists(model_path):
        raise FileNotFoundError(
            f"Model file not found: {model_path}\n"
            f"Download from: https://github.com/opencv/opencv_contrib/tree/master/modules/dnn_superres"
        )
    
    files = [f for f in directory.iterdir() if f.suffix.lower() in supported_extensions]
    
    print(f"\nFound {len(files)} file(s) to upscale with OpenCV DNN Super Resolution")
    print(f"Model: {model_path} | Scale: {scale}x")
    print("="*60)
    
    # Load model once for all images (efficiency)
    try:
        sr_obj: Any = cv2.dnn_superres.DnnSuperResImpl_create()  # type: ignore[attr-defined]
        sr = cast(Any, sr_obj)
        sr.readModel(model_path)
        sr.setModel("lapsrn", scale)
        print(f"✓ Model loaded successfully")
    except Exception as e:
        print(f"✗ Failed to load model: {str(e)}")
        return results
    
    for idx, file in enumerate(files, 1):
        print(f"\n[{idx}/{len(files)}] Processing: {file.name}")
        print("-"*60)
        
        try:
            # Load input image
            img_result = cv2.imread(str(file))
            if img_result is None:
                raise ValueError(f"Failed to load image: {file.name}")
            
            img = cast(np.ndarray[Any, np.dtype[np.uint8]], img_result)
            shape_attr: Any = getattr(img, "shape", (0, 0, 0))
            input_shape = cast(tuple[int, int, int], shape_attr)
            input_height: int = input_shape[0]
            input_width: int = input_shape[1]
            
            print(f"  Input size: {input_width}x{input_height}")
            
            # Upscale image
            upscaled_result: Any = sr.upsample(img)
            upscaled = cast(np.ndarray[Any, np.dtype[np.uint8]], upscaled_result)
            
            # Extract dimensions
            upscaled_shape_attr: Any = getattr(upscaled, "shape", (0, 0, 0))
            output_shape = cast(tuple[int, int, int], upscaled_shape_attr)
            output_height: int = output_shape[0]
            output_width: int = output_shape[1]
            
            # Save result
            output_filename = f"{file.stem}_upscaled_x{scale}{file.suffix}"
            output_file = output_path / output_filename
            cv2.imwrite(str(output_file), upscaled)
            
            print(f"  Output size: {output_width}x{output_height}")
            print(f"  Saved: {output_filename}")
            
            results[file.name] = {
                "success": True,
                "input_path": str(file),
                "output_path": str(output_file),
                "input_size": (input_width, input_height),
                "output_size": (output_width, output_height),
                "scale_factor": scale,
                "message": f"Successfully upscaled from {input_width}x{input_height} to {output_width}x{output_height}",
            }
            print(f"✓ Success")
            
        except Exception as e:
            print(f"✗ Error: {str(e)}")
            results[file.name] = {
                "success": False,
                "input_path": str(file),
                "output_path": None,
                "input_size": None,
                "output_size": None,
                "scale_factor": scale,
                "message": f"Error: {str(e)}",
            }
    
    print("\n" + "="*60)
    print("BATCH UPSCALING COMPLETE")
    print("="*60)
    
    successful = sum(1 for r in results.values() if r.get("success", False))
    failed = len(files) - successful
    
    print(f"\nProcessed: {len(files)} file(s)")
    print(f"Successful: {successful}")
    print(f"Failed: {failed}")
    print(f"Output directory: {output_dir}")
    
    return results


# Example usage (single image):
# Make sure LapSRN_x2.pb model file is in the notebook directory
# Download from: https://github.com/opencv/opencv_contrib/tree/master/modules/dnn_superres

# Uncomment to run single image upscale:
# result = upscale_image("./valid_id4.png", "./valid_id4-upscaled.png")
# print(result['message'])

# Uncomment to run batch upscale:
results = upscale_images_batch("../tmp/s0-nocenter-100/GRAY", "../tmp/s0-nocenter-100/UPSCALED", model_path="LapSRN_x2.pb", scale=2)
print(f"\nUpscaled {sum(1 for r in results.values() if r.get('success'))} images successfully")



Found 7 file(s) to upscale with OpenCV DNN Super Resolution
Model: LapSRN_x2.pb | Scale: 2x
✓ Model loaded successfully

[1/7] Processing: valid_id1_sig_1_1763187513397_sharp_gray1.png
------------------------------------------------------------
  Input size: 359x237
  Output size: 718x474
  Saved: valid_id1_sig_1_1763187513397_sharp_gray1_upscaled_x2.png
✓ Success

[2/7] Processing: valid_id2_sig_1_1763187520058_sharp_gray1.png
------------------------------------------------------------
  Input size: 337x239
  Output size: 718x474
  Saved: valid_id1_sig_1_1763187513397_sharp_gray1_upscaled_x2.png
✓ Success

[2/7] Processing: valid_id2_sig_1_1763187520058_sharp_gray1.png
------------------------------------------------------------
  Input size: 337x239
  Output size: 674x478
  Saved: valid_id2_sig_1_1763187520058_sharp_gray1_upscaled_x2.png
✓ Success

[3/7] Processing: valid_id3_sig_1_1763187536532_sharp_gray1.png
------------------------------------------------------------
  Input s

## Color/Ink Mask + Classic CV

Use when the signature ink is clearly black or blue and the local background isn’t too busy.

In [5]:
def extract_signature_simple(input_path, outdir="outputs_simple",
                             ink_hint="black",
                             bilateral=(9,75,75), clahe=(2.0,(8,8)),
                             unsharp=(1.2,1.5), adaptive=(35,10),
                             open_k=3, close_k=3, min_area=400):
    out = Path(outdir); out.mkdir(parents=True, exist_ok=True)
    bgr = cv2.imread(input_path); gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    # 1) Color/ink mask
    if ink_hint=="black":
        mask_color = cv2.inRange(gray, 0, 80)
    else:  # blue example
        hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
        mask_color = cv2.inRange(hsv, np.array([90,50,20]), np.array([140,255,255]))
    mask_color = cv2.morphologyEx(mask_color, cv2.MORPH_OPEN,
                                  cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3)))

    # 2) Edge-preserving denoise
    d,sC,sS = bilateral
    denoised = cv2.bilateralFilter(gray, d, sC, sS)

    # 3) CLAHE
    clip, tiles = clahe
    contrasted = cv2.createCLAHE(clipLimit=clip, tileGridSize=tiles).apply(denoised)

    # 4) Unsharp
    amt, rad = unsharp
    ksz = max(3, int(2*round(rad)+1))
    blurred = cv2.GaussianBlur(contrasted, (ksz, ksz), rad)
    sharp = cv2.addWeighted(contrasted, 1+amt, blurred, -amt, 0)

    # 5) Adaptive threshold (Gaussian)
    block, C = adaptive
    binary = cv2.adaptiveThreshold(sharp, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY, block, C)

    # keep only regions overlapping the color mask
    ink_bin = cv2.bitwise_and(binary, binary, mask=mask_color)

    # 6) Morphological clean
    openK = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_k,open_k))
    closeK = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(close_k,close_k))
    cleaned = cv2.morphologyEx(ink_bin, cv2.MORPH_OPEN, openK, iterations=1)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, closeK, iterations=1)

    # Largest component
    inv = 255 - cleaned
    n, labels, stats, _ = cv2.connectedComponentsWithStats(inv, 8)
    if n>1:
        idx = 1+np.argmax(stats[1:, cv2.CC_STAT_AREA])
        final = 255 - ((labels==idx).astype(np.uint8)*255)
    else:
        final = cleaned

    # Transparent PNG (+ cropped)
    alpha = (final==0).astype(np.uint8)*255
    rgba = cv2.cvtColor(final, cv2.COLOR_GRAY2RGBA); rgba[:,:,3]=alpha
    stem = Path(input_path).stem
    p = out/f"{stem}_signature_transparent.png"; cv2.imwrite(str(p), rgba)

    # Auto-crop
    ys,xs = np.where(alpha>0)
    if ys.size:
        y0,y1,x0,x1 = ys.min(), ys.max()+1, xs.min(), xs.max()+1
        cv2.imwrite(str(out/f"{stem}_signature_transparent_cropped.png"), rgba[y0:y1, x0:x1])

# Example
extract_signature_simple('valid_id2.png', ink_hint='black')


## GrabCut (Robust Segmentation)

Use when the ID background is busy (micro‑text, guilloches, holograms). We auto‑initialize GrabCut with a coarse “ink‑likelihood” mask; you can optionally pass a rectangle or provide manual hints if needed.

In [6]:
def extract_signature_grabcut(input_path, outdir="outputs_grabcut",
                              rect_padding=10, iters=5, min_area=400):
    out = Path(outdir); out.mkdir(parents=True, exist_ok=True)
    bgr = cv2.imread(input_path); gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    h,w = gray.shape

    # Ink-likelihood for seeding (dark OR edges)
    dark = cv2.inRange(gray, 0, 85); edges = cv2.Canny(gray, 80, 180)
    ink_like = cv2.bitwise_or(dark, edges)
    ink_like = cv2.morphologyEx(ink_like, cv2.MORPH_OPEN,
                                cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3)))

    # Rectangle around detected region
    ys,xs = np.where(ink_like>0)
    rect = (max(0,xs.min()-rect_padding), max(0,ys.min()-rect_padding),
            min(w-1, xs.max()+rect_padding)-max(0,xs.min()-rect_padding),
            min(h-1, ys.max()+rect_padding)-max(0,ys.min()-rect_padding)) if ys.size else (0,0,w-1,h-1)

    # GrabCut
    mask = np.zeros((h,w), np.uint8)
    bg, fg = np.zeros((1,65),np.float64), np.zeros((1,65),np.float64)
    cv2.grabCut(bgr, mask, rect, bg, fg, iters, cv2.GC_INIT_WITH_RECT)  # OpenCV GrabCut

    # Foreground mask + refinement
    m = np.where((mask==cv2.GC_FGD)|(mask==cv2.GC_PR_FGD), 255, 0).astype('uint8')
    m = cv2.bitwise_and(m, ink_like)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3))
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k); m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k)

    # Remove tiny blobs
    cnts,_ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    keep = np.zeros_like(m); [cv2.drawContours(keep,[c],-1,255,-1) for c in cnts if cv2.contourArea(c)>=min_area]
    final = 255 - keep  # black ink, white bg

    # Transparent PNG (+ cropped)
    alpha = (final==0).astype(np.uint8)*255
    rgba = cv2.cvtColor(final, cv2.COLOR_GRAY2RGBA); rgba[:,:,3]=alpha
    stem = Path(input_path).stem
    p = out/f"{stem}_signature_transparent.png"; cv2.imwrite(str(p), rgba)

    ys,xs = np.where(alpha>0)
    if ys.size:
        y0,y1,x0,x1 = ys.min(), ys.max()+1, xs.min(), xs.max()+1
        cv2.imwrite(str(out/f"{stem}_signature_transparent_cropped.png"), rgba[y0:y1, x0:x1])

# Example
extract_signature_grabcut('valid_id2.png')


In [ ]:
import cv2
import numpy as np
from pathlib import Path

def enhance_signature(
    input_path: str,
    output_dir: str = "outputs",
    bilateral_d=9, bilateral_sigmaColor=75, bilateral_sigmaSpace=75,
    clahe_clip=2.0, clahe_tilegrid=(8, 8),
    unsharp_amount=1.2, unsharp_radius=1.5,  # radius -> Gaussian sigma
    adaptive_block=35, adaptive_C=10,
    open_kernel=3, close_kernel=3,
    save_intermediate=True
):
    """
    Enhance a scanned handwritten signature: denoise -> CLAHE -> unsharp -> threshold -> morphology.

    Parameters are tuned for typical A4 scans; adjust for your image if needed.
    """
    # ---- I/O ----
    img_bgr = cv2.imread(input_path)
    if img_bgr is None:
        raise FileNotFoundError(f"Could not read: {input_path}")
    outdir = Path(output_dir); outdir.mkdir(parents=True, exist_ok=True)

    # Convert to grayscale
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # ---- 1) Edge-preserving denoising (Bilateral) ----
    denoised = cv2.bilateralFilter(gray, d=bilateral_d,
                                   sigmaColor=bilateral_sigmaColor,
                                   sigmaSpace=bilateral_sigmaSpace)

    # (Alternative: Non-local Means if the scan is very noisy)
    # denoised = cv2.fastNlMeansDenoising(gray, None, h=10, templateWindowSize=7, searchWindowSize=21)

    # ---- 2) Local contrast boost (CLAHE) ----
    clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=clahe_tilegrid)
    contrasted = clahe.apply(denoised)

    # ---- 3) Unsharp masking (Gaussian blur + weighted add) ----
    # sharpened = original + amount * (original - blurred)
    ksize = max(3, int(2 * round(unsharp_radius) + 1))  # odd kernel size from radius
    blurred = cv2.GaussianBlur(contrasted, (ksize, ksize), unsharp_radius)
    sharpened = cv2.addWeighted(contrasted, 1 + unsharp_amount, blurred, -unsharp_amount, 0)

    # ---- 4) Adaptive thresholding (Gaussian) ----
    # Produces a crisp black-on-white signature
    bin_img = cv2.adaptiveThreshold(
        sharpened, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY,
        blockSize=adaptive_block, C=adaptive_C
    )

    # ---- 5) Morphology: clean speckles & strengthen strokes ----
    # Opening: remove tiny noise; Closing: connect micro gaps
    open_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_kernel, open_kernel))
    close_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_kernel, close_kernel))
    cleaned = cv2.morphologyEx(bin_img, cv2.MORPH_OPEN, open_k, iterations=1)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, close_k, iterations=1)

    # ---- Optional: produce a transparent PNG (signature only) ----
    # Treat white background as transparent alpha
    alpha = (cleaned == 0).astype(np.uint8) * 255  # ink -> opaque (black), white -> transparent
    rgba = cv2.cvtColor(cleaned, cv2.COLOR_GRAY2RGBA)
    rgba[:, :, 3] = alpha

    # ---- Save results ----
    stem = Path(input_path).stem
    cv2.imwrite(str(outdir / f"{stem}_gray.png"), gray) if save_intermediate else None
    cv2.imwrite(str(outdir / f"{stem}_denoised.png"), denoised) if save_intermediate else None
    cv2.imwrite(str(outdir / f"{stem}_clahe.png"), contrasted) if save_intermediate else None
    cv2.imwrite(str(outdir / f"{stem}_unsharp.png"), sharpened) if save_intermediate else None
    cv2.imwrite(str(outdir / f"{stem}_binary.png"), bin_img) if save_intermediate else None
    cv2.imwrite(str(outdir / f"{stem}_cleaned.png"), cleaned)
    cv2.imwrite(str(outdir / f"{stem}_transparent.png"), rgba)

    return {
        "gray": gray, "denoised": denoised, "clahe": contrasted,
        "unsharp": sharpened, "binary": bin_img, "cleaned": cleaned, "transparent_rgba": rgba
    }

# Example usage:
enhance_signature("valid_id2.png", output_dir="outputs_simple")

{'gray': array([[161, 160, 160, ..., 154, 154, 153],
        [163, 161, 160, ..., 151, 151, 151],
        [163, 162, 161, ..., 149, 150, 150],
        ...,
        [152, 152, 153, ..., 109, 108, 106],
        [154, 154, 155, ..., 109, 108, 106],
        [156, 156, 158, ..., 110, 109, 107]], shape=(199, 291), dtype=uint8),
 'denoised': array([[161, 161, 162, ..., 150, 150, 151],
        [160, 161, 161, ..., 151, 151, 151],
        [159, 159, 160, ..., 152, 152, 152],
        ...,
        [155, 155, 156, ..., 107, 107, 106],
        [155, 155, 156, ..., 107, 107, 107],
        [155, 155, 156, ..., 108, 108, 108]], shape=(199, 291), dtype=uint8),
 'clahe': array([[182, 182, 184, ..., 149, 149, 152],
        [179, 182, 182, ..., 152, 152, 152],
        [176, 176, 179, ..., 155, 155, 155],
        ...,
        [188, 188, 190, ..., 130, 130, 127],
        [188, 188, 190, ..., 130, 130, 130],
        [188, 188, 190, ..., 133, 133, 133]], shape=(199, 291), dtype=uint8),
 'unsharp': array([[184

In [11]:
# pip install scikit-image
from skimage.filters import threshold_sauvola
import numpy as np

def sauvola_binarize(gray_img, window_size=25, k=0.2):
    # Sauvola threshold map
    t = threshold_sauvola(gray_img, window_size=window_size, k=k)
    return (gray_img > t).astype(np.uint8) * 255

# Example: Use the results from enhance_signature() above
result = enhance_signature("valid_id2.png", output_dir="outputs_simple")
sharpened = result["unsharp"]
sauvola_bin = sauvola_binarize(sharpened, window_size=33, k=0.2)

# Apply morphological cleaning
open_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
cleaned = cv2.morphologyEx(sauvola_bin, cv2.MORPH_OPEN, open_k)

# Save result
cv2.imwrite("outputs_simple/valid_id2_sauvola.png", cleaned)

True

Pipeline A — Color/Ink Mask + Classic CV (Transparent PNG)

In [24]:
import cv2
import numpy as np
from pathlib import Path
from typing import Any

def extract_signature_simple(
    input_path: str,
    output_dir: str = "outputs_simple",
    ink_hint: str = "auto",      # "auto" | "black" | "blue"
    bilateral_d=9, bilateral_sigmaColor=75, bilateral_sigmaSpace=75,
    clahe_clip=2.0, clahe_tiles=(8, 8),
    unsharp_amount=1.2, unsharp_radius=1.5,
    adaptive_block=35, adaptive_C=10,
    min_area=500,                # discard tiny blobs
    open_kernel=3, close_kernel=3,
    save_intermediate=True
):
    """
    Extract a handwritten signature from an ID scan using color/ink masking + thresholding.
    Produces a transparent PNG with ink opaque and background transparent.
    
    Args:
        input_path: Path to input image file
        output_dir: Directory to save output images
        ink_hint: Ink color hint ("auto", "black", or "blue")
        bilateral_d: Diameter for bilateral filter
        bilateral_sigmaColor: Sigma color for bilateral filter
        bilateral_sigmaSpace: Sigma space for bilateral filter
        clahe_clip: CLAHE clip limit
        clahe_tiles: CLAHE tile grid size
        unsharp_amount: Unsharp masking amount
        unsharp_radius: Unsharp masking radius
        adaptive_block: Adaptive threshold block size
        adaptive_C: Adaptive threshold constant
        min_area: Minimum contour area to keep
        open_kernel: Morphological opening kernel size
        close_kernel: Morphological closing kernel size
        save_intermediate: Whether to save intermediate processing steps
    
    Returns:
        Tuple of (rgba_image, final_binary_image)
    """
    outdir = Path(output_dir); outdir.mkdir(parents=True, exist_ok=True)
    bgr = cv2.imread(input_path)
    if bgr is None: raise FileNotFoundError(f"Cannot read: {input_path}")
    stem = Path(input_path).stem

    # 1) Initial ink mask by color
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    # --- Sharpen using a kernel (Laplacian-like) ---
    # This kernel enhances center pixel and subtracts neighbors to increase contrast.
    sharpen_kernel = np.array([[0, -1,  0],
                            [-1,  5, -1],
                            [0, -1,  0]], dtype=np.float32)

    sharp_gray = cv2.filter2D(gray, ddepth=-1, kernel=sharpen_kernel)

    # Parameters for unsharp masking
    gaussian_radius = 1.5  # Sigma for Gaussian blur; typical range 0.5–2.0
    gaussian_kernel_size = (0, 0)  # (0,0) uses sigma only; or use e.g. (5,5)
    amount = 1.5          # Sharpening strength; typical range 0.5–2.0

    # --- Blur to get low-frequency component ---
    blur = cv2.GaussianBlur(gray, ksize=gaussian_kernel_size, sigmaX=gaussian_radius)

    # --- Unsharp mask: sharpened = gray + amount * (gray - blur) ---
    mask = cv2.subtract(gray, blur)
    sharp_gray1 = cv2.add(gray, cv2.multiply(mask, amount))

    # --- Clip to valid range [0, 255] and cast ---
    sharp_gray1 = np.clip(sharp_gray1, 0, 255).astype(np.uint8)

    if ink_hint == "black":
        # Dark pixels likely ink
        mask_color = cv2.inRange(gray, 0, 70)
    elif ink_hint == "blue":
        # Typical blue pen H ranges ~ 90-140 in HSV; adjust for your scans
        lower = np.array([90, 50, 20], dtype=np.uint8)
        upper = np.array([140, 255, 255], dtype=np.uint8)
        mask_color = cv2.inRange(hsv, lower, upper)
    else:
        # Auto: combine darkness + edge density
        edges = cv2.Canny(gray, 80, 180)
        dark = cv2.inRange(gray, 0, 80)
        mask_color = cv2.bitwise_and(dark, edges)

    # Light cleanup of color mask
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask_color = cv2.morphologyEx(mask_color, cv2.MORPH_OPEN, k, iterations=1)

    # 2) Edge-preserving denoise (bilateral)  — preserves strokes
    denoised = cv2.bilateralFilter(gray, d=bilateral_d,
                                   sigmaColor=bilateral_sigmaColor,
                                   sigmaSpace=bilateral_sigmaSpace)

    # 3) Local contrast (CLAHE)
    clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=clahe_tiles)
    contrasted = clahe.apply(denoised)

    # 4) Unsharp mask
    ksize = max(3, int(2 * round(unsharp_radius) + 1))
    blurred = cv2.GaussianBlur(contrasted, (ksize, ksize), unsharp_radius)
    sharpened = cv2.addWeighted(contrasted, 1 + unsharp_amount, blurred, -unsharp_amount, 0)

    # 5) Adaptive threshold (Gaussian)
    bin_img = cv2.adaptiveThreshold(
        sharpened, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY,
        blockSize=adaptive_block, C=adaptive_C
    )
    # keep only regions overlapping color mask (reduces background text/patterns)
    ink_bin = cv2.bitwise_and(bin_img, bin_img, mask=mask_color)

    # 6) Morphology cleanup & largest component selection
    open_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_kernel, open_kernel))
    close_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_kernel, close_kernel))
    cleaned = cv2.morphologyEx(ink_bin, cv2.MORPH_OPEN, open_k, iterations=1)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, close_k, iterations=1)

    # Keep largest connected component (likely the signature)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(255 - cleaned, connectivity=8)
    # stats[:, cv2.CC_STAT_AREA] sorted descending
    areas = stats[1:, cv2.CC_STAT_AREA] if num_labels > 1 else np.array([])
    if areas.size > 0:
        largest_idx = 1 + np.argmax(areas)
        mask_largest = (labels == largest_idx).astype(np.uint8) * 255
        final = 255 - mask_largest
    else:
        final = cleaned

    # Remove tiny blobs by area threshold
    contours, _ = cv2.findContours(255 - final, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    keep = np.zeros_like(final)
    for c in contours:
        if cv2.contourArea(c) >= min_area:
            cv2.drawContours(keep, [c], -1, color=255, thickness=-1)
    final = 255 - keep

    # 7) Transparent PNG (ink = opaque, background = transparent)
    alpha = (final == 0).astype(np.uint8) * 255
    rgba = cv2.cvtColor(final, cv2.COLOR_GRAY2RGBA)
    rgba[:, :, 3] = alpha

    # Save
    if save_intermediate:
    #     cv2.imwrite(str(outdir / f"{stem}_gray_color.png"), gray)
    #     cv2.imwrite(str(outdir / f"{stem}_sharp_gray.png"), sharp_gray)
        cv2.imwrite(str(outdir / f"{stem}_sharp_gray1.png"), sharp_gray1)
    #     cv2.imwrite(str(outdir / f"{stem}_mask_color.png"), mask_color)
    #     cv2.imwrite(str(outdir / f"{stem}_denoised.png"), denoised)
    #     cv2.imwrite(str(outdir / f"{stem}_clahe.png"), contrasted)
    #     cv2.imwrite(str(outdir / f"{stem}_unsharp.png"), sharpened)
    #     cv2.imwrite(str(outdir / f"{stem}_binary.png"), bin_img)
    #     cv2.imwrite(str(outdir / f"{stem}_ink_bin.png"), ink_bin)
    #     cv2.imwrite(str(outdir / f"{stem}_cleaned.png"), cleaned)
    # cv2.imwrite(str(outdir / f"{stem}_signature_transparent.png"), rgba)

    return rgba, final


def extract_signatures_batch(
    directory_path: str = "../tmp",
    output_dir: str = "outputs_simple",
    ink_hint: str = "auto",
    bilateral_d=9, bilateral_sigmaColor=75, bilateral_sigmaSpace=75,
    clahe_clip=2.0, clahe_tiles=(8, 8),
    unsharp_amount=1.2, unsharp_radius=1.5,
    adaptive_block=35, adaptive_C=10,
    min_area=500,
    open_kernel=3, close_kernel=3,
    save_intermediate=True
) -> dict[str, Any]:
    """
    Batch process all images in a directory to extract signatures.
    Uses color/ink masking + thresholding to produce transparent PNG outputs.
    
    Args:
        directory_path: Path to directory containing images
        output_dir: Directory to save extracted signatures
        ink_hint: Ink color hint ("auto", "black", or "blue")
        bilateral_d: Diameter for bilateral filter
        bilateral_sigmaColor: Sigma color for bilateral filter
        bilateral_sigmaSpace: Sigma space for bilateral filter
        clahe_clip: CLAHE clip limit
        clahe_tiles: CLAHE tile grid size
        unsharp_amount: Unsharp masking amount
        unsharp_radius: Unsharp masking radius
        adaptive_block: Adaptive threshold block size
        adaptive_C: Adaptive threshold constant
        min_area: Minimum contour area to keep
        open_kernel: Morphological opening kernel size
        close_kernel: Morphological closing kernel size
        save_intermediate: Whether to save intermediate processing steps
    
    Returns:
        Dictionary mapping filenames to extraction results
    
    Example:
        >>> results = extract_signatures_batch("../tmp", "outputs_simple", ink_hint="black")
        >>> print(f"Processed {len(results)} images")
    """
    results: dict[str, Any] = {}
    supported_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}
    
    directory = Path(directory_path)
    if not directory.exists():
        raise FileNotFoundError(f"Directory not found: {directory_path}")
    
    # Create output directory
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    files = [f for f in directory.iterdir() if f.suffix.lower() in supported_extensions]
    
    print(f"\nFound {len(files)} file(s) to process with Color/Ink Mask Pipeline")
    print(f"Ink hint: {ink_hint} | Output: {output_dir}")
    print("="*60)
    
    for idx, file in enumerate(files, 1):
        print(f"\n[{idx}/{len(files)}] Processing: {file.name}")
        print("-"*60)
        
        try:
            # Extract signature using single-file function
            rgba, final = extract_signature_simple(
                input_path=str(file),
                output_dir=output_dir,
                ink_hint=ink_hint,
                bilateral_d=bilateral_d,
                bilateral_sigmaColor=bilateral_sigmaColor,
                bilateral_sigmaSpace=bilateral_sigmaSpace,
                clahe_clip=clahe_clip,
                clahe_tiles=clahe_tiles,
                unsharp_amount=unsharp_amount,
                unsharp_radius=unsharp_radius,
                adaptive_block=adaptive_block,
                adaptive_C=adaptive_C,
                min_area=min_area,
                open_kernel=open_kernel,
                close_kernel=close_kernel,
                save_intermediate=save_intermediate
            )
            
            # Get output file info
            output_filename = f"{file.stem}_signature_transparent.png"
            output_file = output_path / output_filename
            
            # Calculate signature dimensions
            ys, xs = np.where(rgba[:, :, 3] > 0)
            if ys.size > 0:
                sig_width = xs.max() - xs.min() + 1
                sig_height = ys.max() - ys.min() + 1
            else:
                sig_width = sig_height = 0
            
            print(f"  Output: {output_filename}")
            print(f"  Signature size: {sig_width}x{sig_height}")
            print(f"✓ Success")
            
            results[file.name] = {
                "success": True,
                "input_path": str(file),
                "output_path": str(output_file),
                "output_filename": output_filename,
                "signature_size": (sig_width, sig_height),
                "message": f"Successfully extracted signature ({sig_width}x{sig_height})",
            }
            
        except Exception as e:
            print(f"✗ Error: {str(e)}")
            results[file.name] = {
                "success": False,
                "input_path": str(file),
                "output_path": None,
                "output_filename": None,
                "signature_size": None,
                "message": f"Error: {str(e)}",
            }
    
    print("\n" + "="*60)
    print("BATCH SIGNATURE EXTRACTION COMPLETE")
    print("="*60)
    
    successful = sum(1 for r in results.values() if r.get("success", False))
    failed = len(files) - successful
    
    print(f"\nProcessed: {len(files)} file(s)")
    print(f"Successful: {successful}")
    print(f"Failed: {failed}")
    print(f"Output directory: {output_dir}")
    
    return results


# Example usage:

# Single image processing:
# rgba, final = extract_signature_simple("valid_id2.png", output_dir="outputs_simple")

# Batch processing:
results = extract_signatures_batch("../tmp/s0-nocenter-100", "../tmp/s0-nocenter-100/GRAY", ink_hint="black")
print(f"\nExtracted {sum(1 for r in results.values() if r.get('success'))} signatures successfully")



Found 7 file(s) to process with Color/Ink Mask Pipeline
Ink hint: black | Output: ../tmp/s0-nocenter-100/GRAY

[1/7] Processing: valid_id1_sig_1_1763187513397.png
------------------------------------------------------------
  Output: valid_id1_sig_1_1763187513397_signature_transparent.png
  Signature size: 359x237
✓ Success

[2/7] Processing: valid_id2_sig_1_1763187520058.png
------------------------------------------------------------
  Output: valid_id2_sig_1_1763187520058_signature_transparent.png
  Signature size: 337x239
✓ Success

[3/7] Processing: valid_id3_sig_1_1763187536532.png
------------------------------------------------------------
  Output: valid_id3_sig_1_1763187536532_signature_transparent.png
  Signature size: 324x238
✓ Success

[4/7] Processing: valid_id4_sig_1_1763187543878.png
------------------------------------------------------------
  Output: valid_id4_sig_1_1763187543878_signature_transparent.png
  Signature size: 399x218
✓ Success

[5/7] Processing: valid_

Pipeline B — GrabCut (Robust Segmentation, Transparent PNG)

In [10]:
import cv2
import numpy as np
from pathlib import Path

def extract_signature_grabcut(
    input_path: str,
    output_dir: str = "outputs_grabcut",
    auto_rect=True,
    rect_padding=10,       # pixels around auto rect
    iterations=5,
    min_area=500,
    save_intermediate=True
):
    """
    Robust extraction using GrabCut foreground segmentation seeded by an ink-likelihood mask.
    Outputs transparent PNG with signature strokes opaque.
    """
    outdir = Path(output_dir); outdir.mkdir(parents=True, exist_ok=True)
    bgr = cv2.imread(input_path)
    if bgr is None: raise FileNotFoundError(f"Cannot read: {input_path}")
    stem = Path(input_path).stem

    h, w = bgr.shape[:2]
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    # 1) Ink-likelihood (darkness + edges); robust across backgrounds
    dark = cv2.inRange(gray, 0, 85)
    edges = cv2.Canny(gray, 80, 180)
    ink_like = cv2.bitwise_or(dark, edges)
    ink_like = cv2.morphologyEx(ink_like, cv2.MORPH_OPEN,
                                cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3)), iterations=1)

    # 2) Build initial mask & rectangle for GrabCut
    mask = np.zeros((h, w), np.uint8)  # 0=background, 1=foreground, 2=prob bg, 3=prob fg
    ys, xs = np.where(ink_like > 0)
    if ys.size > 0 and auto_rect:
        x0, y0 = max(0, xs.min()-rect_padding), max(0, ys.min()-rect_padding)
        x1, y1 = min(w-1, xs.max()+rect_padding), min(h-1, ys.max()+rect_padding)
        rect = (x0, y0, x1-x0, y1-y0)
        mode = cv2.GC_INIT_WITH_RECT
    else:
        # fallback: whole image as rect
        rect = (0, 0, w-1, h-1)
        mode = cv2.GC_INIT_WITH_RECT

    bgdModel = np.zeros((1,65), np.float64)
    fgdModel = np.zeros((1,65), np.float64)

    # 3) Run GrabCut
    cv2.grabCut(bgr, mask, rect, bgdModel, fgdModel, iterations, mode)  # OpenCV impl of GrabCut [9](https://docs.opencv.org/4.x/d3/d47/group__imgproc__segmentation.html)

    # Convert mask to 0/255: foreground (1 or 3) → 255; background (0 or 2) → 0
    mask2 = np.where((mask==cv2.GC_FGD) | (mask==cv2.GC_PR_FGD), 255, 0).astype('uint8')

    # 4) Optional refinement: intersect with ink_like to remove non‑signature foreground
    mask2 = cv2.bitwise_and(mask2, ink_like)

    # 5) Post clean (morphology + area filter)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    mask2 = cv2.morphologyEx(mask2, cv2.MORPH_OPEN, k, iterations=1)
    mask2 = cv2.morphologyEx(mask2, cv2.MORPH_CLOSE, k, iterations=1)

    contours, _ = cv2.findContours(mask2, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    keep = np.zeros_like(mask2)
    for c in contours:
        if cv2.contourArea(c) >= min_area:
            cv2.drawContours(keep, [c], -1, color=255, thickness=-1)
    mask2 = keep

    # 6) Invert to binary signature (white background, black ink)
    final = 255 - mask2

    # 7) Transparent PNG
    alpha = (final == 0).astype(np.uint8) * 255
    rgba = cv2.cvtColor(final, cv2.COLOR_GRAY2RGBA)
    rgba[:, :, 3] = alpha

    # Save
    if save_intermediate:
        cv2.imwrite(str(outdir / f"{stem}_ink_like.png"), ink_like)
        cv2.imwrite(str(outdir / f"{stem}_grabcut_mask.png"), mask2)
        cv2.imwrite(str(outdir / f"{stem}_grabcut_cleaned.png"), final)
    cv2.imwrite(str(outdir / f"{stem}_signature_transparent.png"), rgba)


extract_signature_grabcut("valid_id2.png", output_dir="outputs_grabcut")

## Notes

### ⚡ Serverless Compatibility
**This notebook is SERVERLESS COMPATIBLE** following Azure Functions best practices:
- ✅ **All processing is in-memory by default** - No file system writes required
- ✅ **Optional file saving** - Controlled by `SAVE_CROPS` environment variable
- ✅ **Results always in memory** - `image_bytes` and `image_array` always available
- ✅ **Compatible with Azure Functions** - Can run in serverless/ephemeral environments

**To enable saving crops to tmp/ folder:**
```python
# Set in your .env file
SAVE_CROPS=true
```

**Default behavior (serverless mode):**
```python
# All operations stay in-memory
opencv_results = crop_signatures_opencv("../tmp", "../tmp")
# Access in-memory data from results dictionary
```

### Requirements
- Python packages: opencv-python, PyMuPDF (fitz), numpy, pillow, python-dotenv
- Environment variables in `.env` file:
  - `SAVE_CROPS` - Set to `true` to save cropped images to tmp/ (optional, default: `false`)

### Features
1. **OpenCV Signature Detection**
   - Contour-based detection using adaptive thresholding
   - Filters by area and aspect ratio (1.5-5.0 typical for signatures)
   - Returns top 3 signatures by area
   - In-memory processing by default

2. **Image Upscaling**
   - Uses OpenCV DNN Super Resolution
   - LapSRN model (good for text edges)
   - Supports 2x, 4x, 8x scaling
   - Download models from OpenCV contrib repository

### Tips
- **PDFs are automatically converted** to images at 200 DPI
- **Padding** is added around crops for better visibility
- Adjust `PDF_RENDER_DPI` and `PADDING_PIXELS` for your needs
- **Serverless mode**: Keep `SAVE_CROPS=false` for production/serverless environments
- **Local development**: Set `SAVE_CROPS=true` to save debug images to tmp/

### Performance
- All processing is done in-memory for maximum speed
- Batch processing reduces overhead
- **In-memory processing** is faster and serverless-compatible

### Related Files
- Implementation in Azure Functions: `function_app.py` (see signature extraction functions)
- Other notebooks: `signature_extraction_openai.ipynb`, `signature_extraction_mistral_ai.ipynb`